# **Projeto IA Dália  - Análise do Dataset**

## **Informações:**

Nos notebooks anteriores, realizamos uma análise exploratória inicial e, após a detecção de anomalias, executamos uma etapa para a limpeza da base de dados.

> URL:
>
> https://www.kaggle.com/datasets/csafrit2/maternal-health-risk-data

### **Importando Bibliotecas e Materiais:**

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import boxplot as bp
import pingouin as pg
import itertools

from scipy.stats import mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### **Abrindo Dataset:**

In [3]:
df_clean = pd.read_csv("../data/processed/maternal_health_clean.csv")

df_clean.head(10)

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.00,98.0,86,high risk
1,35,140,90,13.00,98.0,70,high risk
2,29,90,70,8.00,100.0,80,high risk
3,30,140,85,7.00,98.0,70,high risk
4,35,120,60,6.10,98.0,76,low risk
5,23,140,80,7.01,98.0,70,high risk
6,23,130,70,7.01,98.0,78,mid risk
7,35,85,60,11.00,102.0,86,high risk
8,32,120,90,6.90,98.0,70,mid risk
9,42,130,80,18.00,98.0,70,high risk


### **Verificação Inicial:**

In [4]:
df_clean["RiskLevel"].value_counts()

df_clean["RiskLevel"].value_counts(normalize=True) * 100

RiskLevel
low risk     51.662971
high risk    24.833703
mid risk     23.503326
Name: proportion, dtype: float64

### **Estatísticas por Nível de Risco:**

In [5]:
df_clean.groupby("RiskLevel").mean(numeric_only=True)

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
RiskLevel,,,,,,
high risk,33.732143,119.491071,81.535714,11.165357,99.228571,76.482143
low risk,27.369099,105.373391,72.721030,7.198841,98.362232,73.042918
mid risk,28.537736,112.405660,74.886792,7.893585,98.858491,73.896226


### **Análise Descritiva:**

In [6]:
df_clean.groupby("RiskLevel").agg({
    "Age": ["mean", "median"],
    "SystolicBP": ["mean", "median"],
    "DiastolicBP": ["mean", "median"],
    "BS": ["mean", "median"],
    "BodyTemp": ["mean", "median"],
    "HeartRate": ["mean", "median"]
})

Age         SystolicBP        DiastolicBP                BS  \
                mean median        mean median        mean median       mean   
RiskLevel                                                                      
high risk  33.732143   33.5  119.491071  120.0   81.535714   80.0  11.165357   
low risk   27.369099   22.0  105.373391  100.0   72.721030   75.0   7.198841   
mid risk   28.537736   25.5  112.405660  120.0   74.886792   77.5   7.893585   

                   BodyTemp         HeartRate         
          median       mean median       mean median  
RiskLevel                                             
high risk   11.0  99.228571   98.0  76.482143   77.0  
low risk     7.2  98.362232   98.0  73.042918   70.0  
mid risk     7.0  98.858491   98.0  73.896226   76.0

### **Teste ANOVA:**

In [8]:
variaveis = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate"
]

for variavel in variaveis:
    grupos = [
        df_clean[df_clean["RiskLevel"] == nivel][variavel]
        for nivel in ["low risk", "mid risk", "high risk"]
    ]

    stat, p = f_oneway(*grupos)

    print(f"{variavel}: F = {stat:.3f}, p = {p:.5f}")

Age: F = 8.525, p = 0.00023
SystolicBP: F = 27.148, p = 0.00000
DiastolicBP: F = 16.690, p = 0.00000
BS: F = 114.152, p = 0.00000
BodyTemp: F = 16.201, p = 0.00000
HeartRate: F = 8.193, p = 0.00032


> **Como Intrepretar?**
>
> p < 0,05  → diferença estatisticamente significativa
>
> p ≥ 0,05  → não temos evidência suficiente de diferença

### **Teste pós - hoc de Tukey:**

In [10]:
for variavel in variaveis:
    tukey = pairwise_tukeyhsd(
        endog=df_clean[variavel],
        groups=df_clean["RiskLevel"],
        alpha=0.05
    )

    print(f"\n===== {variavel} =====")
    print(tukey)


===== Age =====
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
  group1   group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
high risk low risk   -6.363 0.0002 -10.025  -2.701   True
high risk mid risk  -5.1944 0.0135 -9.5102 -0.8786   True
 low risk mid risk   1.1686 0.7419 -2.5627     4.9  False
---------------------------------------------------------

===== SystolicBP =====
   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
  group1   group2  meandiff p-adj   lower    upper  reject
----------------------------------------------------------
high risk low risk -14.1177    0.0 -18.6952 -9.5402   True
high risk mid risk  -7.0854  0.006 -12.4801 -1.6907   True
 low risk mid risk   7.0323 0.0013   2.3681 11.6964   True
----------------------------------------------------------

===== DiastolicBP =====
   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
  group1   group2  meandiff p-adj   lower    upper  reje

> **Como Intrepretar?**
>
> `reject`
>
> `True` → diferença estatisticamente significativa
>
> `False` → não encontramos diferença significativa entre aqueles dois grupos

**Conclusão:**

| Feature         | High × Low | High × Mid | Low × Mid |
| --------------- | :--------: | :--------: | :-------: |
| **Age**         |      ✅     |      ✅     |     ❌     |
| **SystolicBP**  |      ✅     |      ✅     |     ✅     |
| **DiastolicBP** |      ✅     |      ✅     |     ❌     |
| **BS**          |      ✅     |      ✅     |     ✅     |
| **BodyTemp**    |      ✅     |      ❌     |     ✅     |
| **HeartRate**   |      ✅     |      ✅     |     ❌     |
